In [ ]:
import numpy as np
from patsy import dmatrix
from math import factorial

def permutation_entropy(signal, d=3, t=1):
    n = len(signal)
    indices = np.arange(d) * t
    embedded_vectors = np.array([signal[i + indices] for i in range(n - (d - 1) * t)])
    permutations = [tuple(np.argsort(v)) for v in embedded_vectors]
    counts = {}
    for p in permutations:
        counts[p] = counts.get(p, 0) + 1
    probs = np.array(list(counts.values())) / len(permutations)
    if len(probs) == 0: return 0.0
    npe = -np.sum(probs * np.log(probs)) / np.log(factorial(d))
    return npe

def numerical_trace():
    np.random.seed(42)
    # Sine wave to ensure diversity in ordinal patterns
    x = np.linspace(0, 2*np.pi, 10)
    y = np.sin(x) + np.random.normal(0, 0.05, 10)
    
    # 1. Input
    print("--- Step 1: Input Data (Sine Wave) ---")
    print(np.round(y, 3), "\n")

    # 2. B-Spline Fit
    X_mat = dmatrix("bs(x, df=4, degree=3, include_intercept=True)", {"x": x})
    beta = np.linalg.lstsq(X_mat, y, rcond=None)[0]
    y_hat = np.dot(X_mat, beta)
    print("--- Step 2: B-Spline Fit (y_hat) ---")
    print(np.round(y_hat, 3), "\n")

    # 3. Centered Residuals
    residuals = y - y_hat
    residuals -= np.mean(residuals)
    print("--- Step 3: Centered Residuals ---")
    print(np.round(residuals, 3), "\n")

    # 4. Bootstrap
    B = 5
    boot_matrix = np.zeros((B, 10))
    for b in range(B):
        resample_idx = np.random.choice(len(residuals), size=len(residuals), replace=True)
        boot_matrix[b] = y_hat + residuals[resample_idx]
    print("--- Step 4: Bootstrap Matrix (5x10) ---")
    print(np.round(boot_matrix, 3), "\n")

    # 5. NPE Calculation
    pe_values = np.array([permutation_entropy(row, d=3, t=1) for row in boot_matrix])
    print("--- Step 5: PE Values ---")
    print(np.round(pe_values, 4), "\n")

    # 6. Two-Sided Control Limits
    lcl, ucl = np.percentile(pe_values, [2.5, 97.5])
    print("--- Step 6: Two-Sided Control Limits ---")
    print(f"LCL (2.5th): {np.round(lcl, 4)} | UCL (97.5th): {np.round(ucl, 4)}")

if __name__ == "__main__":
    numerical_trace()

In [ ]:
def print_transformation_roadmap():
    # Define the roadmap data
    roadmap = [
        ["1", "Data Collection", "{x_i, y_i}", "Sampling n points", "Vector (n x 1)"],
        ["2", "B-Spline Fit", "Raw Profile", "LS estimate y_hat = X*beta", "Smooth Vector (n x 1)"],
        ["3", "Residuals", "y and y_hat", "epsilon = y - y_hat (Centered)", "Noise Vector (n x 1)"],
        ["4", "Bootstrapping", "y_hat and epsilon", "Resample (y_hat + epsilon*)", "Matrix (B x n)"],
        ["5", "PE Computation", "B Profiles", "Ordinal Pattern Encoding", "Vector (B x 1)"],
        ["6", "Control Limits", "B PE values", "2.5th and 97.5th Percentiles", "Scalar (LCL, UCL)"]
    ]

    # Define headers
    headers = ["Step", "Operation", "Input Data", "Transformation", "Output Data"]

    # Calculate column widths
    col_widths = [max(len(str(item)) for item in col) for col in zip(*([headers] + roadmap))]
    col_widths = [w + 2 for w in col_widths]  # Add padding

    # Create format string
    fmt = "".join(f"{{:<{w}}}" for w in col_widths)

    # Print table
    print(fmt.format(*headers))
    print("-" * sum(col_widths))
    for row in roadmap:
        print(fmt.format(*row))

if __name__ == "__main__":
    print_transformation_roadmap()